In [1]:
# ============================================================
# CELL 1 — Install & Imports
# ============================================================
!pip install xgboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files

from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, explained_variance_score)
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded!")

✅ All libraries loaded!


In [2]:
# ============================================================
# CELL 2 — Banner
# ============================================================
display(HTML("""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@700;900&family=Rajdhani:wght@400;600&display=swap');
  .banner {
    background: linear-gradient(135deg, #0f0c29, #302b63, #24243e);
    border-radius: 16px; padding: 40px 30px; text-align: center;
    border: 1px solid rgba(167,139,250,0.4);
    box-shadow: 0 0 40px rgba(167,139,250,0.2);
    margin-bottom: 10px;
  }
  .banner h1 {
    font-family: 'Orbitron', monospace; font-size: 1.8em; font-weight: 900;
    background: linear-gradient(90deg, #a78bfa, #7c3aed);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
    margin: 0 0 8px 0; letter-spacing: 2px;
  }
  .banner p { font-family: 'Rajdhani', sans-serif; color: #94a3b8; font-size: 1.1em; }
  .badge {
    display:inline-block;
    background: rgba(167,139,250,0.15); border: 1px solid rgba(167,139,250,0.4);
    border-radius: 8px; padding: 6px 16px; margin: 4px;
    font-family: 'Rajdhani', sans-serif; color: #a78bfa; font-size: 0.9em;
  }
</style>
<div class="banner">
  <h1>🚀 MODEL 1 — Multi-Output Gradient Boosting Regressor</h1>
  <p>Predicts all subject finals simultaneously using internal marks</p>
  <div>
    <span class="badge">📊 Multi-Output Regression</span>
    <span class="badge">🌳 Gradient Boosting</span>
    <span class="badge">📈 All Subjects at Once</span>
    <span class="badge">🎯 MAE · RMSE · R² · CV</span>
  </div>
</div>
"""))




In [3]:
# ============================================================
# CELL 3 — Subject Configuration
# ============================================================
sem1_pairs = [
    ("Operating System mid term cia(30)",              "Operating System Final (70)"),
    ("Data Structures and Algorithms MidSem cia (30)", "Data Structures and Algorithms Final (70)"),
    ("maths  MidSem (30)",                             "maths Final (70)"),
    ("DBMS MidSem cia (30)",                           "DBMS Final (70)"),
    ("DS PRACTICAL(15)",                               "DS PRACTICAL FINAL(35)"),
    ("DBMS PRACTICAL (15)",                            "DBMS PRACTICAL FINAL (35)"),
    ("JAVA PRACTICAL (15)",                            "JAVA PRACTICAL FINAL (35)"),
    ("Problem Solving Techniques PRACTICAL (15)",      "Problem Solving Techniques FINAL (35)"),
    ("Java MidSem (15)",                               "Java final (35)")
]

sem2_subjects = [
    "Degn & Analysis of Algorithms (30)",
    "Artificial Intelligence (30)",
    "Computer Networks - Theory (30)",
    "Software Engineering Methodologies - Theory (30)"
]
lab_subjects = [
    "Computer Networks - Lab (15)",
    "Software Engineering Methodologies - Lab (15)",
    "Web Technologies - Lab (15)",
    "Python Lab (15)"
]

def clean_marks(df):
    df = df.copy()
    for col in df.columns:
        df[col] = df[col].astype(str).str.replace("+", "", regex=False).str.strip()
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def grade(c):
    if c >= 9:   return "O"
    elif c >= 8: return "A+"
    elif c >= 7: return "A"
    elif c >= 6: return "B+"
    elif c >= 5: return "B"
    else:        return "F"

print("✅ Configuration ready!")


✅ Configuration ready!


In [4]:
 #============================================================
# CELL 4 — Upload Data
# ============================================================
display(HTML("""
<div style="background:linear-gradient(135deg,#1e293b,#0f172a);border-radius:12px;
            padding:20px;border:1px solid rgba(167,139,250,0.3);margin:10px 0">
  <h3 style="font-family:'Orbitron',monospace;color:#a78bfa;margin:0 0 10px 0;font-size:0.95em">
    📂 STEP 1 — Upload CSV Files
  </h3>
  <p style="font-family:'Rajdhani',sans-serif;color:#94a3b8;margin:0">
    Upload sem1.csv (training) and sem 2.csv (prediction)
  </p>
</div>
"""))

upload_btn1 = widgets.Button(description='📁 Upload sem1.csv',
    style={'button_color': '#7c3aed'},
    layout=widgets.Layout(width='220px', height='42px'))
upload_btn2 = widgets.Button(description='📁 Upload sem 2.csv',
    style={'button_color': '#5b21b6'},
    layout=widgets.Layout(width='220px', height='42px'))
status1 = widgets.Label(value='⏳ Not uploaded')
status2 = widgets.Label(value='⏳ Not uploaded')

sem1, sem2 = None, None

def upload_sem1(b):
    global sem1
    print("📂 Select sem1.csv ↓")
    files.upload()
    sem1 = pd.read_csv("sem1.csv")
    status1.value = f'✅ sem1.csv — {sem1.shape[0]} rows, {sem1.shape[1]} cols'
    print(f"✅ Loaded: {sem1.shape}")

def upload_sem2(b):
    global sem2
    print("📂 Select sem 2.csv ↓")
    files.upload()
    sem2 = pd.read_csv("sem 2.csv")
    status2.value = f'✅ sem2.csv — {sem2.shape[0]} rows, {sem2.shape[1]} cols'
    print(f"✅ Loaded: {sem2.shape}")

upload_btn1.on_click(upload_sem1)
upload_btn2.on_click(upload_sem2)
display(widgets.HBox([upload_btn1, status1]))
display(widgets.HBox([upload_btn2, status2]))




📂 Select sem1.csv ↓


Saving sem1.csv to sem1.csv
✅ Loaded: (41, 40)
📂 Select sem 2.csv ↓


Saving sem 2.csv to sem 2.csv
✅ Loaded: (41, 20)


In [5]:
# ============================================================
# CELL 5 — Train Model 1 (Multi-Output GBR)

rain_btn = widgets.Button(description='🚀 Train Model 1',
    style={'button_color': '#059669'},
    layout=widgets.Layout(width='200px', height='44px'))
train_out = widgets.Output()

# Global model state
M1_model = None
X_test_global = None
y_test_global = None
sem1_inputs_global = None
sem1_outputs_global = None
metrics_global = {}

def train_model(b):
    global M1_model, X_test_global, y_test_global
    global sem1_inputs_global, sem1_outputs_global, metrics_global

    with train_out:
        clear_output()
        if sem1 is None:
            display(HTML('<p style="color:#f87171;font-family:monospace">❌ Upload sem1.csv first!</p>'))
            return

        sem1_c = clean_marks(sem1).fillna(0)
        sem1_inputs  = [p[0] for p in sem1_pairs]
        sem1_outputs = [p[1] for p in sem1_pairs]
        sem1_inputs_global  = sem1_inputs
        sem1_outputs_global = sem1_outputs

        # Check columns
        missing = [c for c in sem1_inputs + sem1_outputs if c not in sem1_c.columns]
        if missing:
            display(HTML(f'<p style="color:#f87171;font-family:monospace">⚠️ Missing columns: {missing[:3]}...</p>'))

        X = sem1_c[[c for c in sem1_inputs  if c in sem1_c.columns]].fillna(0).values
        y = sem1_c[[c for c in sem1_outputs if c in sem1_c.columns]].fillna(0).values

        display(HTML(f'<p style="color:#60a5fa;font-family:monospace">📊 Dataset: {X.shape[0]} students, {X.shape[1]} features → {y.shape[1]} targets</p>'))

        # Train / test split
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
        X_test_global = X_te
        y_test_global = y_te

        display(HTML('<p style="color:#a78bfa;font-family:monospace">🚀 Training Multi-Output GBR (100 estimators per output)...</p>'))

        gbr = GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42,
            subsample=0.8
        )
        M1_model = MultiOutputRegressor(gbr)
        M1_model.fit(X_tr, y_tr)

        y_pred = M1_model.predict(X_te)

        # ── Per-output metrics ──
        per_subject_maes  = []
        per_subject_rmses = []
        per_subject_r2s   = []
        subject_names     = []

        short_names = ["OS","DSA","Maths","DBMS","DS Lab","DBMS Lab","Java Lab","PST Lab","Java"]

        for i in range(y_te.shape[1]):
            mae_i  = mean_absolute_error(y_te[:, i], y_pred[:, i])
            rmse_i = np.sqrt(mean_squared_error(y_te[:, i], y_pred[:, i]))
            r2_i   = r2_score(y_te[:, i], y_pred[:, i])
            per_subject_maes.append(mae_i)
            per_subject_rmses.append(rmse_i)
            per_subject_r2s.append(r2_i)
            subject_names.append(short_names[i] if i < len(short_names) else f"Sub{i+1}")

        # ── Overall metrics ──
        mae_overall  = mean_absolute_error(y_te.flatten(), y_pred.flatten())
        rmse_overall = np.sqrt(mean_squared_error(y_te.flatten(), y_pred.flatten()))
        r2_overall   = r2_score(y_te, y_pred, multioutput='uniform_average')
        ev_overall   = explained_variance_score(y_te, y_pred, multioutput='uniform_average')
        accuracy_pct = (1 - mae_overall / 70) * 100

        metrics_global = {
            "MAE": mae_overall,
            "RMSE": rmse_overall,
            "R²": r2_overall,
            "Explained Variance": ev_overall,
            "Accuracy (%)": accuracy_pct,
            "Per-Subject MAEs": per_subject_maes,
            "Per-Subject RMSEs": per_subject_rmses,
            "Per-Subject R²s": per_subject_r2s,
            "Subject Names": subject_names
        }

        # ── Cross-Validation (single output for speed) ──
        display(HTML('<p style="color:#a78bfa;font-family:monospace">🔁 Running 5-Fold Cross-Validation...</p>'))
        gbr_cv = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
        # CV on first output as representative
        cv_scores = cross_val_score(gbr_cv, X, y[:, 0],
                                    cv=KFold(n_splits=5, shuffle=True, random_state=42),
                                    scoring='neg_mean_absolute_error')
        cv_maes = -cv_scores
        metrics_global["CV MAE Mean"] = cv_maes.mean()
        metrics_global["CV MAE Std"]  = cv_maes.std()

        # ── Display Results ──
        display(HTML(f"""
        <div style="background:linear-gradient(135deg,#1e293b,#0f172a);border-radius:14px;
                    padding:20px 24px;border:1px solid rgba(52,211,153,0.3);margin:12px 0">
          <h3 style="font-family:'Orbitron',monospace;color:#34d399;margin:0 0 14px 0;font-size:0.95em">
            ✅ MODEL 1 — TRAINING COMPLETE
          </h3>
          <div style="display:flex;gap:16px;flex-wrap:wrap;font-family:'Rajdhani',sans-serif">
            <div style="background:rgba(96,165,250,0.1);border-radius:10px;padding:12px 18px;border:1px solid rgba(96,165,250,0.2)">
              <div style="color:#64748b;font-size:0.8em">MAE</div>
              <div style="color:#60a5fa;font-size:1.6em;font-weight:700">{mae_overall:.2f}</div>
            </div>
            <div style="background:rgba(167,139,250,0.1);border-radius:10px;padding:12px 18px;border:1px solid rgba(167,139,250,0.2)">
              <div style="color:#64748b;font-size:0.8em">RMSE</div>
              <div style="color:#a78bfa;font-size:1.6em;font-weight:700">{rmse_overall:.2f}</div>
            </div>
            <div style="background:rgba(52,211,153,0.1);border-radius:10px;padding:12px 18px;border:1px solid rgba(52,211,153,0.2)">
              <div style="color:#64748b;font-size:0.8em">R² Score</div>
              <div style="color:#34d399;font-size:1.6em;font-weight:700">{r2_overall:.4f}</div>
            </div>
            <div style="background:rgba(251,191,36,0.1);border-radius:10px;padding:12px 18px;border:1px solid rgba(251,191,36,0.2)">
              <div style="color:#64748b;font-size:0.8em">Accuracy</div>
              <div style="color:#fbbf24;font-size:1.6em;font-weight:700">{accuracy_pct:.1f}%</div>
            </div>
            <div style="background:rgba(251,146,60,0.1);border-radius:10px;padding:12px 18px;border:1px solid rgba(251,146,60,0.2)">
              <div style="color:#64748b;font-size:0.8em">CV MAE (5-fold)</div>
              <div style="color:#fb923c;font-size:1.6em;font-weight:700">{cv_maes.mean():.2f} ± {cv_maes.std():.2f}</div>
            </div>
          </div>
        </div>"""))

        # ── Per-Subject MAE Chart ──
        fig, axes = plt.subplots(1, 3, figsize=(17, 5))
        fig.patch.set_facecolor('#0f172a')

        # Chart 1: Per-Subject MAE
        ax = axes[0]; ax.set_facecolor('#1e293b')
        colors = plt.cm.plasma(np.linspace(0.3, 0.9, len(subject_names)))
        bars = ax.barh(subject_names, per_subject_maes, color=colors, edgecolor='#0f172a', height=0.6)
        ax.set_title("MAE per Subject", color='white', fontweight='bold', pad=10)
        ax.set_xlabel("MAE", color='#94a3b8')
        ax.tick_params(colors='#94a3b8', labelsize=8)
        ax.spines[:].set_visible(False)
        ax.xaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)
        for bar, val in zip(bars, per_subject_maes):
            ax.text(val+0.1, bar.get_y()+bar.get_height()/2,
                    f'{val:.2f}', va='center', color='white', fontsize=8)

        # Chart 2: Per-Subject R²
        ax = axes[1]; ax.set_facecolor('#1e293b')
        colors2 = plt.cm.viridis(np.linspace(0.3, 0.9, len(subject_names)))
        bars2 = ax.barh(subject_names, per_subject_r2s, color=colors2, edgecolor='#0f172a', height=0.6)
        ax.set_title("R² per Subject", color='white', fontweight='bold', pad=10)
        ax.set_xlabel("R²", color='#94a3b8')
        ax.tick_params(colors='#94a3b8', labelsize=8)
        ax.spines[:].set_visible(False)
        ax.xaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)
        for bar, val in zip(bars2, per_subject_r2s):
            ax.text(max(val,0)+0.01, bar.get_y()+bar.get_height()/2,
                    f'{val:.3f}', va='center', color='white', fontsize=8)

        # Chart 3: CV MAE Fold Distribution
        ax = axes[2]; ax.set_facecolor('#1e293b')
        fold_labels = [f'Fold {i+1}' for i in range(len(cv_maes))]
        ax.bar(fold_labels, cv_maes, color='#a78bfa', edgecolor='#0f172a', width=0.5)
        ax.axhline(cv_maes.mean(), color='#f87171', linestyle='--', linewidth=2, label=f'Mean: {cv_maes.mean():.2f}')
        ax.set_title("5-Fold Cross-Validation MAE", color='white', fontweight='bold', pad=10)
        ax.set_ylabel("MAE", color='#94a3b8')
        ax.tick_params(colors='#94a3b8')
        ax.spines[:].set_visible(False)
        ax.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)
        ax.legend(fontsize=9, facecolor='#1e293b', labelcolor='white')

        plt.suptitle("Model 1 — Multi-Output GBR: Detailed Metrics", color='white', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig("M1_training_metrics.png", dpi=150, bbox_inches='tight', facecolor='#0f172a')
        plt.show()

        # ── Actual vs Predicted scatter ──
        fig2, ax2 = plt.subplots(figsize=(7, 5))
        fig2.patch.set_facecolor('#0f172a'); ax2.set_facecolor('#1e293b')
        ax2.scatter(y_te.flatten(), y_pred.flatten(), alpha=0.5, s=20,
                    color='#a78bfa', edgecolors='none')
        mn, mx = y_te.min(), y_te.max()
        ax2.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect Prediction')
        ax2.set_xlabel("Actual Marks", color='#94a3b8')
        ax2.set_ylabel("Predicted Marks", color='#94a3b8')
        ax2.set_title("Actual vs Predicted — All Subjects", color='white', fontweight='bold', pad=10)
        ax2.tick_params(colors='#94a3b8')
        ax2.spines[:].set_visible(False)
        ax2.legend(fontsize=9, facecolor='#1e293b', labelcolor='white')
        ax2.text(0.05, 0.93, f"R² = {r2_overall:.4f}", transform=ax2.transAxes,
                 color='#34d399', fontsize=10, fontweight='bold')
        plt.tight_layout()
        plt.savefig("M1_actual_vs_predicted.png", dpi=150, bbox_inches='tight', facecolor='#0f172a')
        plt.show()
print("✅ Model 1 training complete!")

rain_btn.on_click(train_model)
display(rain_btn, train_out)


✅ Model 1 training complete!


Button(description='🚀 Train Model 1', layout=Layout(height='44px', width='200px'), style=ButtonStyle(button_co…

Output()

In [6]:
display(HTML("""
<div style="background:linear-gradient(135deg,#1e293b,#0f172a);border-radius:12px;
            padding:20px;border:1px solid rgba(167,139,250,0.3);margin:20px 0 10px 0">
  <h3 style="font-family:'Orbitron',monospace;color:#a78bfa;margin:0 0 6px 0;font-size:0.95em">
    🔮 STEP 3 — Predict Semester 2 Finals
  </h3>
</div>
"""))

predict_btn = widgets.Button(
    description='⚡ Run Predictions',
    style={'button_color': '#7c3aed'},
    layout=widgets.Layout(width='210px', height='44px')
)

predict_out = widgets.Output()
predictions_df = None

def run_predictions(b):
    global predictions_df

    with predict_out:
        clear_output()

        # ===== CHECK =====
        if M1_model is None:
            display(HTML('<p style="color:red">❌ Train model first!</p>'))
            return

        if sem2 is None:
            display(HTML('<p style="color:red">❌ Upload sem2.csv first!</p>'))
            return

        # ===== CLEAN DATA =====
        sem2_c = clean_marks(sem2).fillna(0)
        df = sem2.copy()

        # ===== DEFINE COLUMNS =====
        all_sem2_cols = sem2_subjects + lab_subjects

              # ===== BUILD INPUT MATRIX (FIXED MAPPING) =====
        n = len(sem2_c)
        sem1_inputs = [p[0] for p in sem1_pairs]

        X_pred = np.zeros((n, len(sem1_inputs)))

        # 🔥 CORRECT SUBJECT MAPPING
        mapping = {
            "Operating System mid term cia(30)": "Degn & Analysis of Algorithms (30)",
            "Data Structures and Algorithms MidSem cia (30)": "Artificial Intelligence (30)",
            "maths  MidSem (30)": "Computer Networks - Theory (30)",
            "DBMS MidSem cia (30)": "Software Engineering Methodologies - Theory (30)",
            "DS PRACTICAL(15)": "Computer Networks - Lab (15)",
            "DBMS PRACTICAL (15)": "Software Engineering Methodologies - Lab (15)",
            "JAVA PRACTICAL (15)": "Web Technologies - Lab (15)",
            "Problem Solving Techniques PRACTICAL (15)": "Python Lab (15)",
            "Java MidSem (15)": "Python Lab (15)"
        }

        # ✅ BUILD INPUT PROPERLY
        for i, sem1_col in enumerate(sem1_inputs):
            sem2_col = mapping.get(sem1_col, None)

            if sem2_col and sem2_col in sem2_c.columns:
                X_pred[:, i] = sem2_c[sem2_col].fillna(0).values
            else:
                X_pred[:, i] = 0
        # ===== PREDICT =====
        preds = M1_model.predict(X_pred)

        total_cols = []

        # ===== THEORY SUBJECTS =====
        for i, col in enumerate(sem2_subjects):
            fin_col = col + "_Final_70"
            tot_col = col + "_Total_100"

            p = np.clip(preds[:, i], 0, 70)

            df[fin_col] = p.round(2)

            internal = sem2_c[col].values if col in sem2_c.columns else np.zeros(n)
            df[tot_col] = (internal + p).round(2)

            total_cols.append(tot_col)

        # ===== LAB SUBJECTS =====
        for i, col in enumerate(lab_subjects):
            fin_col = col + "_Final_35"
            tot_col = col + "_Total_50"

            idx = len(sem2_subjects) + i

            p = np.clip(preds[:, idx] * (35/70), 0, 35)

            df[fin_col] = p.round(2)

            internal = sem2_c[col].values if col in sem2_c.columns else np.zeros(n)
            df[tot_col] = (internal + p).round(2)

            total_cols.append(tot_col)

        # ===== FINAL CALCULATIONS =====
        df["Overall_Total_650"] = df[total_cols].sum(axis=1).round(2)
        df["CGPA"] = (df["Overall_Total_650"] / 650 * 10).round(2)
        df["Grade"] = df["CGPA"].apply(grade)
        df["Result"] = df["CGPA"].apply(lambda x: "Pass" if x >= 5 else "Fail")

        # ===== SAVE =====
        predictions_df = df.copy()
        predictions_df.to_csv("M1_Predictions.csv", index=False)

        # ===== SUMMARY =====
        pass_c = (df["Result"] == "Pass").sum()
        fail_c = (df["Result"] == "Fail").sum()

        display(HTML(f"""
        <div style="background:linear-gradient(135deg,#1e293b,#0f172a);
                    padding:15px;border-radius:10px;color:#34d399">
            ✅ Predictions Complete!<br>
            Students: {len(df)}<br>
            Pass: {pass_c} | Fail: {fail_c}<br>
            Avg CGPA: {df["CGPA"].mean():.2f}
        </div>
        """))

        # ===== PREVIEW =====
        name_col = next((c for c in df.columns if "name" in c.lower()), df.columns[0])

        display(df[[name_col, "Overall_Total_650", "CGPA", "Grade", "Result"]]
                .sort_values("CGPA", ascending=False)
                .head(10))

# ===== BUTTON =====
predict_btn.on_click(run_predictions)
display(predict_btn, predict_out)

Button(description='⚡ Run Predictions', layout=Layout(height='44px', width='210px'), style=ButtonStyle(button_…

Output()

In [7]:
display(HTML("""
<div style="background:linear-gradient(135deg,#1e293b,#0f172a);border-radius:12px;
            padding:20px;border:1px solid rgba(167,139,250,0.3);margin:20px 0 10px 0">
  <h3 style="font-family:'Orbitron',monospace;color:#a78bfa;margin:0;font-size:0.95em">
    📊 STEP 4 — Full Metrics Report
  </h3>
</div>
"""))

metrics_btn = widgets.Button(description='📊 Show Metrics',
    style={'button_color': '#0e7490'},
    layout=widgets.Layout(width='200px', height='44px'))
metrics_out = widgets.Output()

def show_metrics(b):
    with metrics_out:
        clear_output()
        if not metrics_global:
            display(HTML('<p style="color:#f87171;font-family:monospace">❌ Train model first!</p>'))
            return

        m = metrics_global
        display(HTML(f"""
        <h3 style="font-family:'Orbitron',monospace;color:#a78bfa;font-size:1em">
          📊 MODEL 1 — COMPLETE METRICS REPORT
        </h3>"""))

        # Overall metrics table
        rows_html = ""
        metric_data = [
            ("MAE (Mean Absolute Error)", f"{m['MAE']:.4f}", "Lower is better | Avg marks off per prediction"),
            ("RMSE (Root Mean Sq. Error)", f"{m['RMSE']:.4f}", "Penalises large errors more than MAE"),
            ("R² Score", f"{m['R²']:.4f}", "1.0 = perfect; explained variance ratio"),
            ("Explained Variance", f"{m['Explained Variance']:.4f}", "Proportion of variance the model explains"),
            ("Accuracy (%)", f"{m['Accuracy (%)']:.2f}%", "(1 - MAE/70) × 100"),
            ("CV MAE Mean (5-fold)", f"{m['CV MAE Mean']:.4f}", "Cross-validated generalisation error"),
            ("CV MAE Std Dev", f"{m['CV MAE Std']:.4f}", "Stability across folds — lower = more stable"),
        ]
        for name, val, desc in metric_data:
            rows_html += f"""
            <tr style="border-bottom:1px solid #0f172a">
              <td style="padding:10px 14px;color:#94a3b8;font-family:'Rajdhani',sans-serif">{name}</td>
              <td style="padding:10px 14px;color:#60a5fa;font-weight:700;font-family:'Orbitron',monospace;font-size:0.95em">{val}</td>
              <td style="padding:10px 14px;color:#64748b;font-family:'Rajdhani',sans-serif;font-size:0.85em">{desc}</td>
            </tr>"""

        display(HTML(f"""
        <table style="width:100%;border-collapse:collapse;background:linear-gradient(135deg,#1e293b,#0f172a);
                      border-radius:12px;overflow:hidden;margin-bottom:20px">
          <thead><tr style="background:#0f172a;color:#64748b;font-family:'Rajdhani',sans-serif;letter-spacing:1px;font-size:0.82em">
            <th style="padding:12px 14px;text-align:left">METRIC</th>
            <th style="padding:12px 14px;text-align:left">VALUE</th>
            <th style="padding:12px 14px;text-align:left">DESCRIPTION</th>
          </tr></thead>
          <tbody>{rows_html}</tbody>
        </table>"""))

        # Per-subject table
        subj_rows = ""
        for i, name in enumerate(m["Subject Names"]):
            subj_rows += f"""
            <tr style="border-bottom:1px solid #0f172a">
              <td style="padding:8px 14px;color:#e2e8f0;font-family:'Rajdhani',sans-serif">{name}</td>
              <td style="padding:8px 14px;color:#60a5fa;font-weight:700">{m['Per-Subject MAEs'][i]:.2f}</td>
              <td style="padding:8px 14px;color:#a78bfa;font-weight:700">{m['Per-Subject RMSEs'][i]:.2f}</td>
              <td style="padding:8px 14px;color:#34d399;font-weight:700">{m['Per-Subject R²s'][i]:.4f}</td>
            </tr>"""

        display(HTML(f"""
        <h4 style="font-family:'Orbitron',monospace;color:#a78bfa;font-size:0.85em">PER-SUBJECT BREAKDOWN</h4>
        <table style="width:100%;border-collapse:collapse;background:linear-gradient(135deg,#1e293b,#0f172a);
                      border-radius:12px;overflow:hidden">
          <thead><tr style="background:#0f172a;color:#64748b;font-family:'Rajdhani',sans-serif;letter-spacing:1px;font-size:0.82em">
            <th style="padding:10px 14px;text-align:left">SUBJECT</th>
            <th style="padding:10px 14px;text-align:left">MAE</th>
            <th style="padding:10px 14px;text-align:left">RMSE</th>
            <th style="padding:10px 14px;text-align:left">R²</th>
          </tr></thead>
          <tbody>{subj_rows}</tbody>
        </table>"""))

        # Feature importance (average across estimators)
        try:
            importances = [est.feature_importances_ for est in M1_model.estimators_]
            avg_imp = np.mean(importances, axis=0)
            short_in = ["OS","DSA","Maths","DBMS","DS Lab","DBMS Lab","Java Lab","PST Lab","Java"]
            fig, ax = plt.subplots(figsize=(8, 4))
            fig.patch.set_facecolor('#0f172a'); ax.set_facecolor('#1e293b')
            labels = short_in[:len(avg_imp)]
            colors_fi = plt.cm.plasma(np.linspace(0.2, 0.9, len(avg_imp)))
            ax.barh(labels, avg_imp, color=colors_fi, edgecolor='#0f172a')
            ax.set_title("Average Feature Importance (across all GBR estimators)",
                         color='white', fontweight='bold', pad=10)
            ax.set_xlabel("Importance", color='#94a3b8')
            ax.tick_params(colors='#94a3b8', labelsize=9)
            ax.spines[:].set_visible(False)
            ax.xaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)
            plt.tight_layout()
            plt.savefig("M1_feature_importance.png", dpi=150, bbox_inches='tight', facecolor='#0f172a')
            plt.show()
        except Exception as e:
            print(f"Feature importance plot skipped: {e}")

metrics_btn.on_click(show_metrics)
display(metrics_btn, metrics_out)


Button(description='📊 Show Metrics', layout=Layout(height='44px', width='200px'), style=ButtonStyle(button_col…

Output()

In [8]:
display(HTML("""
<div style="margin-top:20px">
  <div style="background:linear-gradient(135deg,#1e293b,#0f172a);border-radius:12px;
              padding:16px;border:1px solid rgba(167,139,250,0.3)">
    <h3 style="font-family:'Orbitron',monospace;color:#a78bfa;margin:0 0 8px 0;font-size:0.9em">
      ⬇️ STEP 5 — Download Results
    </h3>
  </div>
</div>
"""))

dl_btn = widgets.Button(description='⬇️ Download CSV',
    style={'button_color': '#b45309'},
    layout=widgets.Layout(width='200px', height='44px'))

def download_all(b):
    try:
        files.download("M1_Predictions.csv")
        print("⬇️ M1_Predictions.csv downloading...")
    except Exception as e:
        print(f"❌ {e}")

dl_btn.on_click(download_all)
display(dl_btn)


Button(description='⬇️ Download CSV', layout=Layout(height='44px', width='200px'), style=ButtonStyle(button_co…

In [9]:
!pip install plotly -q

In [10]:
# ==============================
# 🌟 INTERACTIVE DASHBOARD (NEW CELL)
# ==============================

!pip install plotly -q

import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

dash_btn = widgets.Button(
    description='🌟 Open Interactive Dashboard',
    layout=widgets.Layout(width='300px', height='50px'),
    style={'button_color': '#9333ea'}
)

dash_out = widgets.Output()

def show_dashboard(b):
    with dash_out:
        clear_output()

        # ===== USE EXISTING DATA =====
        try:
            df_dash = predictions_df.copy()
        except:
            print("❌ Run prediction first!")
            return

        # ===== BASIC CALCULATIONS =====
        name_col = next((c for c in df_dash.columns if "name" in c.lower()), df_dash.columns[0])

        topper = df_dash.loc[df_dash["CGPA"].idxmax()]
        avg_cgpa = df_dash["CGPA"].mean()
        pass_count = (df_dash["Result"] == "Pass").sum()
        fail_count = (df_dash["Result"] == "Fail").sum()

        # ===== PREMIUM UI HEADER =====
        display(HTML(f"""
        <div style="background:linear-gradient(135deg,#0f172a,#1e293b);
                    padding:30px;border-radius:20px;
                    box-shadow:0 0 40px rgba(147,51,234,0.4)">

        <h1 style="color:#c084fc;text-align:center;font-size:28px">
        🎓 STUDENT ANALYTICS DASHBOARD
        </h1>

        <div style="display:flex;justify-content:center;gap:25px;margin-top:25px;flex-wrap:wrap">

        <div style="background:#111827;padding:20px;border-radius:15px;width:220px;text-align:center">
            <h4 style="color:#9ca3af">👑 Topper</h4>
            <h2 style="color:#22c55e">{topper[name_col]}</h2>
            <p style="color:#9ca3af">CGPA: {topper["CGPA"]}</p>
        </div>

        <div style="background:#111827;padding:20px;border-radius:15px;width:220px;text-align:center">
            <h4 style="color:#9ca3af">📊 Avg CGPA</h4>
            <h2 style="color:#60a5fa">{avg_cgpa:.2f}</h2>
        </div>

        <div style="background:#111827;padding:20px;border-radius:15px;width:220px;text-align:center">
            <h4 style="color:#9ca3af">✅ Pass</h4>
            <h2 style="color:#22c55e">{pass_count}</h2>
        </div>

        <div style="background:#111827;padding:20px;border-radius:15px;width:220px;text-align:center">
            <h4 style="color:#9ca3af">❌ Fail</h4>
            <h2 style="color:#ef4444">{fail_count}</h2>
        </div>

        </div>
        </div>
        """))

        # ===== INTERACTIVE CHARTS =====

        # 🎯 PIE CHART
        fig1 = px.pie(
            values=[pass_count, fail_count],
            names=["Pass", "Fail"],
            hole=0.5,
            title="🎯 Pass vs Fail"
        )
        fig1.show()

        # 📊 CGPA HISTOGRAM
        fig2 = px.histogram(
            df_dash,
            x="CGPA",
            nbins=10,
            title="📊 CGPA Distribution",
        )
        fig2.show()

        # 🏆 TOP 10 BAR
        top10 = df_dash.sort_values("CGPA", ascending=False).head(10)
        fig3 = px.bar(
            top10,
            x=name_col,
            y="CGPA",
            title="🏆 Top 10 Students",
            text="CGPA"
        )
        fig3.update_traces(textposition='outside')
        fig3.show()

        # 📉 RESULT COUNT
        result_counts = df_dash["Result"].value_counts()
        fig4 = px.bar(
            x=result_counts.index,
            y=result_counts.values,
            title="📉 Result Overview"
        )
        fig4.show()

        # 📋 TABLE
        display(HTML("<h3 style='color:#a78bfa'>🏆 Top 5 Students</h3>"))
        display(top10[[name_col, "CGPA", "Grade"]].head())

        print("✅ Interactive Dashboard Loaded!")

dash_btn.on_click(show_dashboard)
display(dash_btn, dash_out)

Button(description='🌟 Open Interactive Dashboard', layout=Layout(height='50px', width='300px'), style=ButtonSt…

Output()

In [11]:
# ============================================================
# 🌟 CLEAN PROFESSIONAL INPUT UI (NO TEXT CUT + CLEAR MARKS)
# ============================================================

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import numpy as np

display(HTML("""
<div style="background:linear-gradient(135deg,#0f172a,#1e293b);
padding:25px;border-radius:18px;border:1px solid #7c3aed">

<h2 style="color:#c084fc;text-align:center">
🎓 STUDENT PREDICTION SYSTEM
</h2>

<p style="text-align:center;color:#9ca3af">
All inputs clearly show MAX MARKS (No confusion)
</p>
</div>
"""))

# =============================
# 👤 STUDENT DETAILS
# =============================
name = widgets.Text(description="Name:", layout=widgets.Layout(width='600px'))
usn  = widgets.Text(description="USN:", layout=widgets.Layout(width='600px'))

display(name, usn)

# =============================
# 📘 SEM 1 (MODEL INPUT ONLY)
# =============================
display(HTML("<h3 style='color:#a78bfa'>📘 Semester 1 CIA Marks (Used for Prediction)</h3>"))

sem1_info = [
    ("OS CIA (30)", 30),
    ("DSA CIA (30)", 30),
    ("Maths CIA (30)", 30),
    ("DBMS CIA (30)", 30),
    ("DS Lab (15)", 15),
    ("DBMS Lab (15)", 15),
    ("Java Lab (15)", 15),
    ("PST Lab (15)", 15),
    ("Java CIA (15)", 15)
]

sem1_boxes = []

for label, max_marks in sem1_info:
    box = widgets.BoundedFloatText(
        description=f"{label} (Max {max_marks})",
        min=0,
        max=max_marks,
        layout=widgets.Layout(width='600px'),  # 🔥 FIXED WIDTH
        style={'description_width': '350px'}   # 🔥 NO TEXT CUT
    )
    sem1_boxes.append(box)

display(widgets.VBox(sem1_boxes))

# =============================
# 📗 SEM 2 INTERNALS
# =============================
display(HTML("<h3 style='color:#34d399'>📗 Semester 2 Internal Marks</h3>"))

sem2_info = [
    ("ADA (30)", 30),
    ("AI (30)", 30),
    ("CN (30)", 30),
    ("SE (30)", 30),
    ("CN Lab (15)", 15),
    ("SE Lab (15)", 15),
    ("Web Lab (15)", 15),
    ("Python Lab (15)", 15)
]
sem2_boxes = []

for label, max_marks in sem2_info:
    box = widgets.BoundedFloatText(
        description=f"{label} (Max {max_marks})",
        min=0,
        max=max_marks,
        layout=widgets.Layout(width='600px'),
        style={'description_width': '350px'}
    )
    sem2_boxes.append(box)

display(widgets.VBox(sem2_boxes))

# =============================
# 🚀 PREDICT BUTTON
# =============================
btn = widgets.Button(
    description="🚀 Predict Result",
    style={'button_color': '#22c55e'},
    layout=widgets.Layout(width='300px', height='50px')
)

out = widgets.Output()

def predict_full(b):
    with out:
        clear_output()

        if M1_model is None:
            print("❌ Train model first!")
            return

        sem1_values = np.array([b.value for b in sem1_boxes]).reshape(1, -1)
        sem2_values = [b.value for b in sem2_boxes]

        preds = M1_model.predict(sem1_values)[0]

        total = 0

        print(f"\n👤 {name.value} ({usn.value})\n")

        # THEORY
        for i in range(4):
            internal = sem2_values[i]
            final = np.clip(preds[i], 0, 70)
            total_sub = internal + final
            total += total_sub

            print(f"{sem2_info[i][0]}")
            print(f"  {internal}/30 + {final:.2f}/70 = {total_sub:.2f}/100\n")

        # LAB
        for i in range(4):
            idx = 4 + i
            internal = sem2_values[idx]
            final = np.clip(preds[idx]*(35/70), 0, 35)
            total_sub = internal + final
            total += total_sub

            print(f"{sem2_info[idx][0]}")
            print(f"  {internal}/15 + {final:.2f}/35 = {total_sub:.2f}/50\n")

        cgpa = (total / 650) * 10

        def grade(c):
            if c >= 9: return "O"
            elif c >= 8: return "A+"
            elif c >= 7: return "A"
            elif c >= 6: return "B+"
            elif c >= 5: return "B"
            else: return "F"

        display(HTML(f"""
        <div style="background:#111827;padding:20px;border-radius:15px;margin-top:15px">
            <h3 style="color:#22c55e">FINAL RESULT</h3>
            <p>Total: {total:.2f} / 650</p>
            <p>CGPA: {cgpa:.2f}</p>
            <p>Grade: {grade(cgpa)}</p>
        </div>
        """))

btn.on_click(predict_full)

display(btn, out)

Text(value='', description='Name:', layout=Layout(width='600px'))

Text(value='', description='USN:', layout=Layout(width='600px'))

Button(description='🚀 Predict Result', layout=Layout(height='50px', width='300px'), style=ButtonStyle(button_c…

Output()